In [ ]:
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
import torch

In [ ]:
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, id2label=id2label, label2id=label2id, dtype=torch.float32)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
model.base_model.requires_grad_(False)
model.score.requires_grad_(True)
model.eval()   # score has no dropout, so eval on the whole model is fine

function to compute metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
model_feat = "Qwen2515BInstruct"
FEAT_DIR = f"./saved_features/amazon_polarity_content_mean/{model_feat}"

scorer_train_feats = torch.load(f"{FEAT_DIR}/feat_train.pt")
scorer_val_feats = torch.load(f"{FEAT_DIR}/feat_val.pt")
scorer_train_labels = torch.load(f"{FEAT_DIR}/labels_train.pt")
scorer_val_labels = torch.load(f"{FEAT_DIR}/labels_val.pt")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

ds = TensorDataset(scorer_train_feats, scorer_train_labels)
dl = DataLoader(ds, batch_size=256, shuffle=True)
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.score.parameters(), lr=1e-4, weight_decay=0.1)
best_val_loss = float("inf")
patience = 20
epochs_val_plateau = 0

for epoch in range(1000):
    with torch.inference_mode():
        val_logits = model.score(scorer_val_feats)
        val_loss = loss_fn(val_logits, scorer_val_labels)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_score_state = {k: v.clone() for k, v in model.score.state_dict().items()}
            epochs_val_plateau = 0
        else:
            epochs_val_plateau +=1
            if epochs_val_plateau == patience:
                break

        if epoch % 10 == 0:
            print(
                compute_metrics((val_logits.detach().numpy(), scorer_val_labels.detach().numpy())), val_loss.detach().numpy()
            )

    for b_features, b_labels in dl:
        out = model.score(b_features)
        loss = loss_fn(out, b_labels)
        loss.backward()
        opt.step()
        opt.zero_grad()

# keep the head with the lowest validation loss, not the last one
model.score.load_state_dict(best_score_state)
torch.save(best_score_state, f"{FEAT_DIR}/score_head.pt")
print(f"best val loss {best_val_loss:.4f}")


In [ ]:
# score generations as in the loop: one-hot tokens -> Y @ E_J -> head reads the mean state over the response
import numpy as np
device = "cpu"
AGENT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
gen_model = AutoModelForCausalLM.from_pretrained(AGENT_MODEL).to(device).eval()
E_J = model.get_input_embeddings().weight
BASE = "Write a short sentence about public transport. "
INSTRUCTIONS = [
    "Be extremely enthusiastic and positive.",
    "Be strictly neutral and impersonal; use no positive or negative language.",
    "Be extremely critical and negative.",
]
STOP = [tok.convert_tokens_to_ids("<|im_end|>"), tok.eos_token_id]

with torch.inference_mode():
    for ins in INSTRUCTIONS:
        prompt_ids = tok.apply_chat_template(
            [{"role": "user", "content": BASE + ins}],
             add_generation_prompt=True,
             return_tensors="pt"
            )["input_ids"].to(device)
        s_all = []
        for seed in range(20):
            torch.manual_seed(seed)
            ids = gen_model.generate(prompt_ids, do_sample=True, max_new_tokens=32, pad_token_id=tok.eos_token_id)[0, prompt_ids.shape[1]:]
            ids = ids[~torch.isin(ids, torch.tensor(STOP))]
            Y = torch.nn.functional.one_hot(ids, E_J.shape[0]).float()
            h = model.base_model(inputs_embeds=(Y @ E_J)[None]).last_hidden_state[0].mean(0)
            logits = model.score(h)
            s = logits[1] - logits[0]  # log-odds of positive
            s_all.append(s.item())
            print(f"{s.item():+.2f}  {tok.decode(ids)!r}")
        print(f"== {ins}  mean {np.mean(s_all):+.2f}  std {np.std(s_all):.2f}\n")